# Solving Stokes Surface Flow

We want to solve
$$
\begin{alignat}{3}
- \mathbf{P} div_{\Gamma} (E(\mathbf{u}_T)) + \nabla_{\Gamma} p + \alpha \mathbf{u} &= \mathbf{f}_T && \quad \text{on} \Gamma
\\
div_{\Gamma} \mathbf{u}_T &= g && \text{on} \Gamma.
\end{alignat}
$$


In [1]:
import numpy as np
from netgen.csg import *
from netgen.meshing import MeshingStep
from ngsolve import *
from ngsolve.webgui import Draw
ngsglobals.msg_level = 1

# Generate mesh and plot it
center = Pnt(0,0,0)
R = 1.0
geo = CSGeometry()
geo.Add(Sphere(center, R))
maxh = 1.0
mesh = Mesh(geo.GenerateMesh(maxh=maxh, perfstepsend=MeshingStep.MESHSURFACE))
#   We use a superparametric approximation to ensure 
#   optimal convergence of solution also in w.r.t to normal component 
order_k = 3
order_g = order_k+1
mesh.Curve(order_g)
Draw(mesh)

 Start Findpoints
 Find edges
 Start Findpoints
 Find edges
 Start Findpoints
 Find edges
 Surface 1 / 1
 Curve elements, order = 4


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

Define geometry related objects and differential operators.

In [ ]:
# Mesh size
h = specialcf.mesh_size

# Normal vector
ns = specialcf.normal(mesh.dim)
# normal and tangential  projection
Qs = OuterProduct(ns, ns)
Ps = Id(mesh.dim) - Qs

# Weingarten map
Bs = Grad(ns)

# TODO: Or should we rather use
# specialcf.Weingarten(mesh.dim)

# Define covariant derivative for vectors
def Nabla_h(u):
    return Ps*Grad(u).Trace()*Ps

# and its corrected version for non-tangential vectors
def Nabla_th(u):
    return Nabla_(u) - InnerProduct(u, ns)*Bs

# Define discrete strain rate tensor
# TODO: Need to clarify meaning/implementation of Grad(u).Trace()
#       for VectorH1 and which projection Ps we actually need.
#       Right now, we safeguard by multiplying from left and right
def E_h(u):
    return Ps*Sym(Grad(u).Trace())*Ps

# Corrected strain rate tensor for non-tangential 
def E_th(u):
    return E_h(u) - InnerProduct(u, ns)*Bs

Define test and trial function spaces and the weak form for the Stokes problem
using the strain rate tensor based formulation.

In [ ]:
V = VectorH1(mesh, order=int(order_k))
Q = H1(mesh, order=int(order_k-1))
N = NumberSpace(mesh)
W = V*Q*N

(u, p, lam), (v, q, mu) = W.TnT()

# Penalty parameter to weakly enforce tangential condition
beta = 1.0

Astar = BilinearForm(W)
# Add elliptic part
Astar += 2*InnerProduct(E_th(u), E_th(v))*ds
Astar += InnerProduct(Ps*u.Trace(), Ps*v.Trace())*ds
# Add incompressibility/pressure related parts
Astar += InnerProduct(grad(p).Trace(), v.Trace())*ds
Astar += InnerProduct(grad(q).Trace(), u.Trace())*ds
# Add global Lagrange multiplier to ensure 0 mean pressure
Astar += (p*mu + q*lam)*ds
Astar.Assemble()
# Penalization of normal component
# TODO:More efficient way would be to compute (u,n)*(v*n) directly
# Test that!
Astar += beta*h**(-2.0)*InnerProduct(Qs*u.Trace(), Qs*v.Trace())*ds


# Tangential test field
f = CF((-y*(1-z)*(1+z), x*(1-z)*(1+z), 0))
l = LinearForm(W)
# NOTE: For non-trivial Dirichlet b.c. remember to include
#       \int_{\partial Gamma} u *n v dE into rhs.
l += InnerProduct(Ps*f.Trace(), v.Trace())*ds
l.Assemble()

Solve and plot solution.

In [ ]:
# Solve problem and write out solution
up_h =  GridFunction(W)
Ainv = Astar.mat.Inverse(freedofs=W.FreeDofs())
up_h.vec.data = Ainv*l.vec

u_h, p_h, _ = up_h.components

Draw(uex_h)
Draw(u_h)
Draw(pex_h)
Draw(p_h)
vtkfile = VTKOutput(mesh, coefs=[u_h, p_h, f], names=["u_h", "p_h", "f"], filename="results_stokes",subdivision=3)
vtkfile.Do(vb=BND)